# Video Eventness Score Visualizer

This notebook allows you to compute and visualize eventness and cluster representativeness scores for videos using pre-extracted frame features. It supports multiple configurable temporal window sizes (`delta_t` in seconds) and displays interactive raw and moving-average-smoothed line charts with an embedded video player for manual inspection.

### Table of Contents
1. **Configuration**: Set default paths and parameters
2. **Utility Functions**: Load video metadata and features
3. **Compute Eventness & Representativeness Scores**: Compute L2 distance over specified `delta_t` windows and K-Means cluster representativeness
4. **Interactive Plotting**: Plot raw/smoothed eventness curves and cluster representativeness using Plotly
5. **Display Video Player**: Embed direct HTML5 video player
6. **Usability API**: Unified visualizer function `visualize_video_eventness`


## 1. Configuration

In [ ]:
import os
import sys

parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

def resolve_path(path):
    """
    Resolve a path from either the workspace root or the notebook directory.
    """
    if path is None:
        return None
    if os.path.exists(path):
        return path
    workspace_relative = os.path.join("..", path)
    if os.path.exists(workspace_relative):
        return workspace_relative
    return path

QA_DATASET_PATH_HINTS = {
    "video-mme": "playground/gt_qa_files/Videomme/val_qa.json",
    "videomme": "playground/gt_qa_files/Videomme/val_qa.json",
    "nextqa": "playground/gt_qa_files/NExTQA/val_qa.json",
    "intentqa": "playground/gt_qa_files/IntentQA/val_qa.json",
    "egoschema": "playground/gt_qa_files/EgoSchema/val_qa.json",
}

# Add dataset-specific LLM answer files here.
# Each file should use the same per-record JSON or JSONL schema as the NextQA file below.
LLM_ANSWER_PATH_HINTS = {
    "video-mme": None,
    "videomme": None,
    "nextqa": "outputs/nextqa/full_cls_new_token_sim/nextqa_ktv_full_cls_new_token_sim_tokens1872.json",
    "intentqa": None,
    "egoschema": None,
}

# Default configuration
# DEFAULT_VIDEO_PATH = "datasets/Video-MME/data/_rC5V5vEprs.mp4"
# DEFAULT_FEATURE_PATH = "ktv/save_tensor/Video-MME/_rC5V5vEprs.pkl"

DEFAULT_VIDEO_PATH = "datasets/NExTQA/videos/2503404966.mp4"
DEFAULT_FEATURE_PATH = "ktv/save_tensor/NExTQA/2503404966.pkl"
DEFAULT_LLM_ANSWER_FILE_PATH = LLM_ANSWER_PATH_HINTS["nextqa"]
DEFAULT_DELTA_T_VALUES = [0.5, 1.0, 2.0, 3.0, 5.0]
DEFAULT_MOVING_AVERAGE_WINDOW = 5  # Number of feature samples to average for smoothing

resolved_vid = resolve_path(DEFAULT_VIDEO_PATH)
resolved_feat = resolve_path(DEFAULT_FEATURE_PATH)
resolved_llm_answers = resolve_path(DEFAULT_LLM_ANSWER_FILE_PATH)
print(f"Default Video Path: {DEFAULT_VIDEO_PATH} (resolved: {resolved_vid}, exists: {os.path.exists(resolved_vid)})")
print(f"Default Feature Path: {DEFAULT_FEATURE_PATH} (resolved: {resolved_feat}, exists: {os.path.exists(resolved_feat)})")
print(
    f"Default LLM Answer File: {DEFAULT_LLM_ANSWER_FILE_PATH} "
    f"(resolved: {resolved_llm_answers}, exists: {resolved_llm_answers is not None and os.path.exists(resolved_llm_answers)})"
)
print(f"Default Moving Average Window: {DEFAULT_MOVING_AVERAGE_WINDOW} feature samples")


Default Video Path: datasets/Video-MME/data/_rC5V5vEprs.mp4 (resolved: ../datasets/Video-MME/data/_rC5V5vEprs.mp4, exists: True)
Default Feature Path: ktv/save_tensor/Video-MME/_rC5V5vEprs.pkl (resolved: ../ktv/save_tensor/Video-MME/_rC5V5vEprs.pkl, exists: True)
Default Moving Average Window: 5 feature samples


## 2. Utility Functions

Functions to load video metadata (FPS, total frame count) and the pre-extracted frame features.

In [2]:
import html
import json
import pickle
import re

import cv2
import numpy as np

def get_video_metadata(video_path):
    """
    Load video FPS and total frame count using OpenCV.
    """
    resolved_path = resolve_path(video_path)
    if not os.path.exists(resolved_path):
        raise FileNotFoundError(f"Video file not found at: {video_path}")

    cap = cv2.VideoCapture(resolved_path)
    if not cap.isOpened():
        raise IOError(f"Failed to open video file: {resolved_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    if fps is None or fps <= 0:
        fps = 1.0

    return fps, total_frames

def load_frame_features(feature_path):
    """
    Load pre-extracted frame features from a pickle file.
    """
    resolved_path = resolve_path(feature_path)
    if not os.path.exists(resolved_path):
        raise FileNotFoundError(f"Feature file not found at: {feature_path}")

    with open(resolved_path, "rb") as f:
        data = pickle.load(f)

    if not isinstance(data, dict):
        raise ValueError(f"Feature pickle file at {resolved_path} does not contain a dictionary.")

    if not data:
        raise ValueError(f"Feature dictionary in {resolved_path} is empty.")

    features = list(data.values())[0]
    return np.asarray(features, dtype=np.float32)

def infer_file_path_from_hints(video_path, feature_path, path_hints):
    """
    Infer a dataset-specific file path from the video or feature path.
    """
    candidate_paths = [video_path]
    if feature_path is not None:
        candidate_paths.append(feature_path)

    for candidate in candidate_paths:
        if candidate is None:
            continue
        normalized = str(candidate).replace(chr(92), "/").lower()
        for marker, file_path in path_hints.items():
            if marker in normalized:
                return file_path
    return None

def infer_qa_file_path(video_path, feature_path=None):
    """
    Infer the QA annotation file from the video or feature path.
    """
    return infer_file_path_from_hints(video_path, feature_path, QA_DATASET_PATH_HINTS)

def infer_llm_answer_file_path(video_path, feature_path=None):
    """
    Infer the dataset-specific LLM answer file from the video or feature path.
    """
    return infer_file_path_from_hints(video_path, feature_path, LLM_ANSWER_PATH_HINTS)

def natural_sort_key(value):
    parts = re.split(r"(\d+)", str(value))
    return [int(part) if part.isdigit() else part.lower() for part in parts]

def normalize_question_text(question):
    if isinstance(question, list):
        return " / ".join(str(item).strip() for item in question if str(item).strip())
    return str(question).strip()

def normalize_record_key(value):
    if value is None:
        return ""
    return str(value).strip()

def load_structured_records(file_path):
    """
    Load either a JSON array/dict file or a JSONL file and return a list of records.
    """
    resolved_path = resolve_path(file_path)
    if resolved_path is None or not os.path.exists(resolved_path):
        raise FileNotFoundError(f"File not found at: {file_path}")

    with open(resolved_path, "r", encoding="utf-8") as f:
        raw_text = f.read().strip()

    if not raw_text:
        return [], resolved_path

    try:
        data = json.loads(raw_text)
    except json.JSONDecodeError:
        records = []
        for line_number, line in enumerate(raw_text.splitlines(), start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"Invalid JSON on line {line_number} of {resolved_path}: {exc}"
                ) from exc
    else:
        if isinstance(data, list):
            records = data
        elif isinstance(data, dict):
            if data and all(isinstance(value, dict) for value in data.values()):
                records = list(data.values())
            else:
                records = [data]
        else:
            raise ValueError(
                f"Unsupported file structure in {resolved_path}: {type(data).__name__}"
            )

    return records, resolved_path

def load_related_questions(video_path, feature_path=None, qa_file_path=None):
    """
    Load every QA entry that matches the selected video.
    """
    resolved_video_path = resolve_path(video_path)
    video_name = os.path.basename(resolved_video_path)
    video_stem = os.path.splitext(video_name)[0]

    inferred_qa_path = qa_file_path or infer_qa_file_path(video_path, feature_path)
    if inferred_qa_path is None:
        return [], None

    records, resolved_qa_path = load_structured_records(inferred_qa_path)

    related_questions = []
    for sample in records:
        sample_video_name = os.path.basename(str(sample.get("video_name", "")))
        sample_video_stem = os.path.splitext(sample_video_name)[0]
        if sample_video_name == video_name or sample_video_stem == video_stem:
            related_questions.append(sample)

    related_questions.sort(key=lambda sample: natural_sort_key(sample.get("question_id", "")))
    return related_questions, resolved_qa_path

def load_llm_answers(video_path, feature_path=None, llm_answer_file_path=None):
    """
    Load dataset-specific LLM answers keyed by question ID.
    """
    inferred_answer_path = llm_answer_file_path or infer_llm_answer_file_path(video_path, feature_path)
    if inferred_answer_path is None:
        return {}, None

    records, resolved_answer_path = load_structured_records(inferred_answer_path)
    answers_by_question_id = {}

    for sample in records:
        for key in (sample.get("question_id"), sample.get("id")):
            normalized_key = normalize_record_key(key)
            if normalized_key:
                answers_by_question_id[normalized_key] = sample

    return answers_by_question_id, resolved_answer_path

def get_llm_answer_text(sample, llm_answers_by_question_id):
    if not llm_answers_by_question_id:
        return None

    for key in (sample.get("question_id"), sample.get("id")):
        normalized_key = normalize_record_key(key)
        if not normalized_key:
            continue
        matched_sample = llm_answers_by_question_id.get(normalized_key)
        if not matched_sample:
            continue
        for field in ("pred", "prediction", "llm_answer", "model_answer", "answer"):
            value = matched_sample.get(field)
            if value not in (None, ""):
                return str(value).strip()
    return None

def display_related_questions(
    related_questions,
    video_path,
    qa_source_path,
    show_answers=False,
    llm_answers_by_question_id=None,
    llm_answer_source_path=None,
):
    """
    Render all questions associated with the selected video inside the notebook.
    """
    from IPython.display import HTML, display

    video_name = os.path.basename(resolve_path(video_path))
    if not related_questions:
        print(f"[INFO] No related questions found for {video_name} in {qa_source_path}.")
        return

    question_cards = []
    for index, sample in enumerate(related_questions, start=1):
        question_id = html.escape(str(sample.get("question_id", index)))
        question_text = html.escape(normalize_question_text(sample.get("question", "")))
        candidates = sample.get("candidates") or []

        options_html = ""
        if candidates:
            option_items = "".join(
                f"<li style='margin-bottom: 4px;'>{html.escape(str(option))}</li>"
                for option in candidates
            )
            options_html = (
                f"<ol style='margin: 8px 0 0 20px; padding: 0;'>"
                f"{option_items}"
                f"</ol>"
            )

        answer_sections = []
        if show_answers and sample.get("answer") is not None:
            answer_sections.append(
                f"<p style='margin: 10px 0 0 0; color: #0f5132;'>"
                f"<strong>Ground Truth:</strong> {html.escape(str(sample.get('answer')))}"
                f"</p>"
            )

        if show_answers:
            llm_answer = get_llm_answer_text(sample, llm_answers_by_question_id or {})
            if llm_answer is not None:
                answer_sections.append(
                    f"<p style='margin: 6px 0 0 0; color: #0a58ca;'>"
                    f"<strong>LLM Answer:</strong> {html.escape(llm_answer)}"
                    f"</p>"
                )

        answer_html = "".join(answer_sections)

        question_cards.append(
            f"""
            <div style="border: 1px solid #d0d7de; border-radius: 10px; padding: 14px 16px; margin: 10px 0; background: #f8fafc;">
                <div style="font-size: 13px; color: #57606a; margin-bottom: 6px;">Question {index} · ID {question_id}</div>
                <div style="font-size: 15px; font-weight: 600; color: #24292f;">{question_text}</div>
                {options_html}
                {answer_html}
            </div>
            """
        )

    qa_label = html.escape(os.path.basename(qa_source_path))
    if show_answers:
        if llm_answer_source_path is not None:
            llm_label = html.escape(os.path.basename(llm_answer_source_path))
            answers_note = f"Ground-truth and LLM answers shown when available. LLM source: {llm_label}."
        else:
            answers_note = "Ground-truth answers shown. LLM answer file not configured for this dataset yet."
    else:
        answers_note = "Answers hidden."

    display(
        HTML(
            f"""
            <div style="margin: 12px 0 20px 0;">
                <div style="font-size: 18px; font-weight: 700; margin-bottom: 6px;">Questions related to {html.escape(video_name)}</div>
                <div style="font-size: 13px; color: #57606a; margin-bottom: 12px;">Source: {qa_label}. {len(related_questions)} question(s) found. {answers_note}</div>
                {''.join(question_cards)}
            </div>
            """
        )
    )


## 3. Compute Eventness Scores

Calculate eventness score for each frame. For a configurable window size `delta_t` in seconds:
$$eventness(t) = ||f(t) - f(t - \Delta t)||_2 + ||f(t + \Delta t) - f(t)||_2$$

Where $f(t)$ is the visual feature at time $t$.

In [3]:
def get_frame_indices(total_frames, max_frames):
    """
    Return evenly spaced frame indices used during feature extraction,
    matching the feature extraction pipeline in extract_frame_features.py.
    """
    if total_frames <= max_frames:
        return np.arange(total_frames)
    return np.linspace(0, total_frames - 1, max_frames, dtype=int)

def compute_eventness_scores(features, timestamps, delta_t):
    """
    Compute eventness score for each frame given a temporal window delta_t (in seconds).
    Handles boundary conditions by pinning to the start/end frames.
    """
    n = len(features)
    scores = np.zeros(n, dtype=np.float32)
    if n <= 1:
        return scores

    for t in range(n):
        t_time = timestamps[t]

        left_time = t_time - delta_t
        left_idx = np.searchsorted(timestamps, left_time)
        if left_idx >= n:
            left_idx = n - 1
        elif left_idx > 0:
            if abs(timestamps[left_idx - 1] - left_time) < abs(timestamps[left_idx] - left_time):
                left_idx = left_idx - 1

        right_time = t_time + delta_t
        right_idx = np.searchsorted(timestamps, right_time)
        if right_idx >= n:
            right_idx = n - 1
        elif right_idx > 0:
            if abs(timestamps[right_idx - 1] - right_time) < abs(timestamps[right_idx] - right_time):
                right_idx = right_idx - 1

        diff_left = np.linalg.norm(features[t] - features[left_idx])
        diff_right = np.linalg.norm(features[right_idx] - features[t])
        scores[t] = diff_left + diff_right

    return scores

def compute_moving_average(scores, window_size):
    """
    Smooth a 1D score array with a centered moving average.
    Uses edge padding to avoid artificially lowering the first/last values.
    """
    window_size = int(window_size)
    if window_size <= 1 or len(scores) == 0:
        return np.asarray(scores, dtype=np.float32).copy()

    kernel = np.ones(window_size, dtype=np.float32) / window_size
    left_pad = window_size // 2
    right_pad = window_size - 1 - left_pad
    padded_scores = np.pad(np.asarray(scores, dtype=np.float32), (left_pad, right_pad), mode="edge")
    return np.convolve(padded_scores, kernel, mode="valid").astype(np.float32)

def compute_representativeness_scores(features, num_clusters=5, score_normalizer="minmax"):
    """
    Compute cluster representativeness score for each frame.
    Uses KMeans clustering and normalizes the negative distance to cluster centers.
    """
    from ktv.methods.clustering import perform_clustering
    from ktv.methods.temporal_chain import normalize_scores

    labels, centers, r_cluster = perform_clustering(features, num_clusters, clustering_method="kmeans")

    if score_normalizer is None or score_normalizer == "raw":
        normalized_scores = r_cluster
    else:
        normalized_scores = normalize_scores(r_cluster, score_normalizer)

    return normalized_scores, labels


## 4. Plotting & Interactive Charts

Create interactive Plotly charts for both the raw eventness scores and a moving-average-smoothed view for easier trend inspection.


In [4]:
import plotly.graph_objects as go
from IPython.display import HTML, display

def display_plotly_figure(fig):
    """
    Render Plotly figures inline with an HTML fallback when mime rendering is unavailable.
    """
    try:
        fig.show()
    except Exception as exc:
        if "nbformat>=4.2.0" not in str(exc):
            raise
        display(HTML(fig.to_html(full_html=False, include_plotlyjs=True)))

def apply_eventness_layout(fig, title, yaxis_title):
    """
    Apply a shared Plotly layout to eventness charts.
    """
    fig.update_layout(
        title=title,
        xaxis_title="Video Time (seconds)",
        yaxis_title=yaxis_title,
        legend_title="Temporal Window",
        hovermode="x unified",
        template="plotly_white",
        margin=dict(l=40, r=40, t=60, b=40),
        legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.99),
    )
    return fig

def plot_eventness_curves(timestamps, all_scores, delta_t_values):
    """
    Plot interactive eventness score curves over time for various delta_t values.
    """
    fig = go.Figure()

    for delta_t in delta_t_values:
        if delta_t in all_scores:
            fig.add_trace(
                go.Scatter(
                    x=timestamps,
                    y=all_scores[delta_t],
                    mode="lines",
                    name=f"delta_t = {delta_t:.1f}s",
                    hovertemplate="Time: %{x:.2f}s<br>Eventness: %{y:.4f}<extra></extra>",
                )
            )

    return apply_eventness_layout(fig, "Eventness Scores Over Time", "Eventness Score")

def plot_smoothed_eventness_curves(timestamps, all_scores, delta_t_values, moving_average_window):
    """
    Plot moving-average-smoothed eventness curves over time for various delta_t values.
    """
    fig = go.Figure()

    for delta_t in delta_t_values:
        if delta_t in all_scores:
            smoothed_scores = compute_moving_average(all_scores[delta_t], moving_average_window)
            fig.add_trace(
                go.Scatter(
                    x=timestamps,
                    y=smoothed_scores,
                    mode="lines",
                    name=f"delta_t = {delta_t:.1f}s",
                    hovertemplate="Time: %{x:.2f}s<br>Smoothed Eventness: %{y:.4f}<extra></extra>",
                )
            )

    title = f"Smoothed Eventness Scores Over Time (Moving Average Window = {moving_average_window} samples)"
    return apply_eventness_layout(fig, title, "Smoothed Eventness Score")

def plot_representativeness_curve(timestamps, scores, labels, num_clusters):
    """
    Plot interactive cluster representativeness score curve over time.
    Colors the markers by their cluster assignment and shows cluster ID in hover.
    """
    fig = go.Figure()
    customdata = [f"Cluster {label}" for label in labels]

    fig.add_trace(
        go.Scatter(
            x=timestamps,
            y=scores,
            mode="lines+markers",
            marker=dict(
                color=labels,
                colorscale="Viridis",
                size=8,
                showscale=True,
                colorbar=dict(
                    title="Cluster ID",
                    tickvals=list(range(num_clusters)),
                    ticktext=[f"C{i}" for i in range(num_clusters)],
                ),
            ),
            line=dict(color="lightgray", width=1.5),
            customdata=customdata,
            hovertemplate="Time: %{x:.2f}s<br>Representativeness: %{y:.4f}<br>%{customdata}<extra></extra>",
            name="Representativeness",
        )
    )

    fig.update_layout(
        title=f"Cluster Representativeness Over Time ({num_clusters} Clusters)",
        xaxis_title="Video Time (seconds)",
        yaxis_title="Representativeness Score",
        template="plotly_white",
        margin=dict(l=40, r=40, t=60, b=40),
    )
    return fig


## 5. Embedded Video Player

In [5]:
from IPython.display import Video, display

def display_video_player(video_path):
    """
    Embed an HTML5 video player in the notebook.
    """
    resolved_path = resolve_path(video_path)
    if not os.path.exists(resolved_path):
        print(f"[ERROR] Cannot embed video player: File not found at {video_path}")
        return

    display(Video(resolved_path, width=640, embed=False, html_attributes="controls"))


## 6. Usability API

A single unified function to load, compute, plot, and display the video, the raw eventness chart, and the smoothed eventness chart with robust error management.


In [6]:
def visualize_video_eventness(
    video_path,
    feature_path,
    delta_t_values,
    moving_average_window=DEFAULT_MOVING_AVERAGE_WINDOW,
    num_clusters=12,
    score_normalizer="minmax",
    qa_file_path=None,
    llm_answer_file_path=None,
    show_answers=False,
):
    """
    Main usability wrapper function to visualize the video player, related QA items,
    ground-truth/LLM answers, and eventness/representativeness charts for a single video.
    """
    print("=== Visualizing Video Eventness ===")
    print(f"Video path: {video_path}")
    print(f"Feature path: {feature_path}")
    print(f"Moving average window: {moving_average_window} feature samples")
    if qa_file_path is not None:
        print(f"QA file override: {qa_file_path}")
    if llm_answer_file_path is not None:
        print(f"LLM answer file override: {llm_answer_file_path}")

    resolved_vid = resolve_path(video_path)
    resolved_feat = resolve_path(feature_path)

    if not os.path.exists(resolved_vid):
        print(f"[ERROR] Video file does not exist: {video_path}")
        return
    if not os.path.exists(resolved_feat):
        print(f"[ERROR] Feature pickle file does not exist: {feature_path}")
        return

    try:
        moving_average_window = int(moving_average_window)
        if moving_average_window < 1:
            raise ValueError("moving_average_window must be at least 1.")

        fps, total_frames = get_video_metadata(resolved_vid)
        print(f"Video Metadata: {total_frames} total frames @ {fps:.2f} FPS (Duration: {total_frames / fps:.2f}s)")

        features = load_frame_features(resolved_feat)
        num_features = len(features)
        print(f"Features Loaded: Shape {features.shape}")

        indices = get_frame_indices(total_frames, num_features)
        timestamps = indices / fps

        if len(timestamps) != num_features:
            raise ValueError(
                f"Timestamp mapping length ({len(timestamps)}) does not match features count ({num_features})."
            )

        all_scores = {}
        for delta_t in delta_t_values:
            all_scores[delta_t] = compute_eventness_scores(features, timestamps, delta_t)

        print("\n--- Interactive Video Player ---")
        display_video_player(resolved_vid)

        print("\n--- Questions Related to This Video ---")
        llm_answers_by_question_id = {}
        resolved_llm_answer_path = None
        if show_answers:
            try:
                llm_answers_by_question_id, resolved_llm_answer_path = load_llm_answers(
                    resolved_vid,
                    feature_path=resolved_feat,
                    llm_answer_file_path=llm_answer_file_path,
                )
            except Exception as exc:
                print(f"[INFO] Could not load LLM answers: {exc}")

        try:
            related_questions, resolved_qa_path = load_related_questions(
                resolved_vid,
                feature_path=resolved_feat,
                qa_file_path=qa_file_path,
            )
        except Exception as exc:
            print(f"[INFO] Could not load related questions: {exc}")
        else:
            if resolved_qa_path is None:
                print("[INFO] No matching QA file was inferred. Pass qa_file_path=... to display related questions.")
            else:
                print(f"Using QA file: {resolved_qa_path}")
                print(f"Found {len(related_questions)} related question(s) for {os.path.basename(resolved_vid)}.")
                if show_answers:
                    if resolved_llm_answer_path is None:
                        print(
                            "[INFO] No LLM answer file was configured for this dataset. "
                            "Populate LLM_ANSWER_PATH_HINTS or pass llm_answer_file_path=... ."
                        )
                    else:
                        print(f"Using LLM answer file: {resolved_llm_answer_path}")
                display_related_questions(
                    related_questions,
                    resolved_vid,
                    resolved_qa_path,
                    show_answers=show_answers,
                    llm_answers_by_question_id=llm_answers_by_question_id,
                    llm_answer_source_path=resolved_llm_answer_path,
                )

        print("\n--- Interactive Eventness Score Chart ---")
        fig = plot_eventness_curves(timestamps, all_scores, delta_t_values)
        display_plotly_figure(fig)

        print("\n--- Smoothed Eventness Score Chart ---")
        smoothed_fig = plot_smoothed_eventness_curves(
            timestamps,
            all_scores,
            delta_t_values,
            moving_average_window,
        )
        display_plotly_figure(smoothed_fig)

        print(
            f"Computing representativeness scores (num_clusters={num_clusters}, normalizer={score_normalizer})..."
        )
        rep_scores, labels = compute_representativeness_scores(
            features,
            num_clusters=num_clusters,
            score_normalizer=score_normalizer,
        )

        print("\n--- Interactive Cluster Representativeness Chart ---")
        rep_fig = plot_representativeness_curve(timestamps, rep_scores, labels, num_clusters)
        display_plotly_figure(rep_fig)

    except Exception as exc:
        print("\n[ERROR] An unexpected error occurred during visualization:")
        print(str(exc))


## 7. Execution Example

In [7]:
# Run visualizer on default paths.
# Set show_answers=False if you want to hide the ground-truth and LLM answers.
visualize_video_eventness(
    DEFAULT_VIDEO_PATH,
    DEFAULT_FEATURE_PATH,
    DEFAULT_DELTA_T_VALUES,
    DEFAULT_MOVING_AVERAGE_WINDOW,
    llm_answer_file_path=DEFAULT_LLM_ANSWER_FILE_PATH,
    show_answers=True,
)


=== Visualizing Video Eventness ===
Video path: datasets/Video-MME/data/_rC5V5vEprs.mp4
Feature path: ktv/save_tensor/Video-MME/_rC5V5vEprs.pkl
Moving average window: 5 feature samples


Video Metadata: 7466 total frames @ 24.00 FPS (Duration: 311.08s)
Features Loaded: Shape (5400, 1024)

--- Interactive Video Player ---



--- Questions Related to This Video ---
Using QA file: ../playground/gt_qa_files/Videomme/val_qa.json
Found 3 related question(s) for _rC5V5vEprs.mp4.



--- Interactive Eventness Score Chart ---



--- Smoothed Eventness Score Chart ---


Computing representativeness scores (num_clusters=12, normalizer=minmax)...


/home/tuanhoang/KTV/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



--- Interactive Cluster Representativeness Chart ---
